In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

csv_path = os.path.join(path, "Q1_data.csv")
df = pd.read_csv(csv_path)


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Delivery_Time")  # we can see that the data is not equally distributed so we need to use stratified kfold for splitting

In [ ]:
# Task 1: Write your code here:
df = df.drop('Order_ID', axis=1)

In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

df["Delivery_Time"] = df["Delivery_Time"].mean()
df["Weather"] = df["Weather"].mode()
df["Traffic_Level"] = df["Traffic_Level"].mode()

df["Time_of_Day"] = df["Time_of_Day"].mode()
df["Courier_Experience_yrs"] = df["Courier_Experience_yrs"].mean()




In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)
df

In [ ]:

# task4
from sklearn.preprocessing import OneHotEncoder #import OneHotEncoder
cat_cols = df.select_dtypes(exclude=["number"]).columns
num_cols = df.select_dtypes(include=["number"]).columns

print('data before encoding:\n', df[cat_cols]) #show before encoding

onehot_encoder = OneHotEncoder(sparse_output=False) # Instantiate OneHotEncoder
data_onehot_encoded = onehot_encoder.fit_transform(df[cat_cols]) # Apply fit_transform to the copied
X_num = df[num_cols].to_numpy()
df = np.hstack([X_num, data_onehot_encoded])

print('\nData after encoding:\n', data_onehot_encoded)


In [ ]:
# Task 5: Write your code here:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# 1. Initialize the StandardScaler
scaler = StandardScaler()

# 2. Fit and Transform the data
# This calculates the mean and std, then scales the data
scaled_data = scaler.fit_transform(df)

# 3. Convert the resulting NumPy array back into a DataFrame
df_scaled = pd.DataFrame(scaled_data, columns=df.columns)

# Display the first few rows
print(df_scaled.head())


In [ ]:
# Task 6: Write your code here:
df.value_counts(normalize=True) # normalize give percentage

In [ ]:
# Separate features and target
X = df_scaled.drop(columns=['target'])
y = df['target']

In [ ]:
# Task 2,3,4,5: Write your code here:
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor

# 1. Define the KFold strategy
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 2. Initialize the Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

# 3. Evaluate using MAE only
# 'neg_mean_absolute_error' returns negative values; we multiply by -1
mae_scores = -cross_val_score(rf_model, X, y, cv=kf, scoring='neg_mean_absolute_error')

# 4. Print the averaged score
print(f"MAE per fold: {mae_scores}")
print(f"Average MAE across all folds: {np.mean(mae_scores):.4f}")

In [ ]:
# Task 1: Write your code here:import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_predict

# 1. Plot Feature Importance
# We fit the model on the full scaled dataset to get final importance scores
rf_model.fit(X_scaled_df, y)

plt.figure(figsize=(10, 6))
importances = rf_model.feature_importances_
feature_names = X.columns
# Sort importances in descending order
indices = np.argsort(importances)[::-1]

sns.barplot(x=importances[indices], y=feature_names[indices], palette='viridis')
plt.title('Feature Importance (Random Forest)')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.tight_layout()
plt.savefig('feature_importance.png')

# 2. Plot Predicted Delivery Time Histogram
# Using cross_val_predict to get "out-of-fold" predictions for the histogram
y_pred = cross_val_predict(rf_model, X_scaled_df, y, cv=kf)

plt.figure(figsize=(10, 6))
sns.histplot(y_pred, kde=True, color='blue', bins=30)
plt.title('Distribution of Predicted Delivery Times')
plt.xlabel('Predicted Delivery Time (mins)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig('predicted_histogram.png')

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: